# EDA - Eksploracyjna Analiza Danych

## Przygotowanie środowiska i wczytanie danych

Import bibliotek, konfiguracja ścieżek i funkcje pomocnicze

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv('../data/raw/produkcja.csv')

def plot_box_hist(df, flag_col, value_col, palette="Set1"):
    fig, axes = plt.subplots(1, 2, figsize=(14,6))
    sns.boxplot(
        x=flag_col,
        y=value_col,
        hue=flag_col,
        data=df,
        palette=palette,
        ax=axes[0],
        legend=False
    )
    axes[0].set_title(f"Wykres pudełkowy {flag_col} vs {value_col}")

    sns.histplot(
        data=df,
        x=value_col,
        hue=flag_col,
        palette=palette,
        ax=axes[1],
        kde=True
    )
    axes[1].set_title(f"Histogram {value_col}")

    plt.tight_layout()
    plt.show()


Inżynieria cech - tworzenie zmiennych pochodnych

In [ ]:
power = (df['Rotational speed [rpm]'] * df['Torque [Nm]'] * (2 * np.pi / 60)).round(2)

df.insert(loc = 8, column = 'Power [W]', value = power)
df.insert(loc = 5, column = 'Temperature difference', value = df['Process temperature [K]'] - df['Air temperature [K]'])

## Profilowanie zbioru - struktura i jakość danych

### Struktura danych

Podgląd pierwszych rekordów

In [ ]:
df.head(10)

Weryfikacja kompletności

In [ ]:
df.isnull().any().any()

Określenie rozmiaru

In [ ]:
df.shape

Identyfikacja typów danych

In [ ]:
df.dtypes

Podstawowe statystyki opisowe

In [ ]:
df.describe()

Identyfikacja duplikatów

In [ ]:
df.duplicated().sum()

In [ ]:
display(df['UDI'].is_unique)
display(df['Product ID'].is_unique)

Identyfikacja nieunikatowych ID

In [ ]:
pd.crosstab(df['Type'], df["Product ID"].str[0])

### Weryfikacja Poprawności Danych

#### Analiza przypadków brzegowych

In [ ]:
display(df[df['RNF']== 1].shape[0])
display(df[df['RNF']== 1])

**Obserwacja:**

Brak przypisania Machine failure dla flagi podtypu RNF

In [ ]:
display(df[((df['TWF'] == 1) | (df['HDF'] == 1) | (df['PWF'] == 1) | (df['OSF'] == 1)| (df['RNF'] == 1)) & (df['Machine failure'] == 0)])
display(df[((df['TWF'] == 1) | (df['HDF'] == 1) | (df['PWF'] == 1) | (df['OSF'] == 1)| (df['RNF'] == 1)) & (df['Machine failure'] == 0)].shape[0])

**Obserwacja:**

Istnieje 18 takich rekordów

##### Analiza parametrów dla RNF

Porównanie podstawowych statystyk opisowych

In [ ]:
display(df[((df['TWF'] == 1) | (df['HDF'] == 1) | (df['PWF'] == 1) | (df['OSF'] == 1)| (df['RNF'] == 1)) & (df['Machine failure'] == 0)].describe())

In [ ]:
df.describe()

Detekcja awarii bez przypisania podtypu

In [ ]:
display(df[((df['TWF'] == 0) & (df['HDF'] == 0) & (df['PWF'] == 0) & (df['OSF'] == 0) & (df['RNF'] == 0)) & (df['Machine failure'] == 1)])
display(df[((df['TWF'] == 0) & (df['HDF'] == 0) & (df['PWF'] == 0) & (df['OSF'] == 0) & (df['RNF'] == 0)) & (df['Machine failure'] == 1)].shape[0])

##### Analiza parametrów dla Machine failure bez wskazania typu awarii

Porównanie podstawowych statystyk opisowych

In [ ]:
anomalie = (df['TWF'] == 0) & (df['HDF'] == 0) & (df['PWF'] == 0) & (df['OSF'] == 0) & (df['RNF'] == 0) & (df['Machine failure'] == 1)
df2 = df[anomalie]
df2.describe()

In [ ]:
df.describe()

**Obserwacja:**

Porównanie podstawowych statystyk opisowych dla 9 anomalnych rekordów wskazuje, że ich średnie parametry znajdują się w granicach normy.
Brak ekstremalnych wartości sugeruje, że te awarie mogą być efektem losowego szumu w etykietach danych, a nie nagłego, fizycznego uszkodzenia komponentu

---

Analiza awarii przy zerowym zużyciu narzędzia

In [ ]:
df[(df['Tool wear [min]' ] == 0) & (df['Machine failure'] == 1)]

## Analiza zależności między parametrami a awarią

Histogramy dla wszystkich numerycznych kolumn

In [ ]:
df.hist(figsize=(12, 10), bins=20, edgecolor='black', alpha=0.7)
plt.tight_layout()
plt.show()

### Analiza struktury awarii w zależności od typu maszyny (L, M, H)

Zestawienie liczby maszyn według typu

In [ ]:
df['Type'].value_counts()

Łączna liczba awarii

In [ ]:
df[df["Machine failure"] == 1].shape[0]

Procentowy udział awarii

In [ ]:
df["Machine failure"].value_counts(normalize=True) * 100

Zestawienie liczby awarii według typu maszyny

In [ ]:
df[df['Machine failure'] == 1]['Type'].value_counts()

Poziom awaryjności w zależności od typu maszyny

In [ ]:
crosstab = pd.crosstab(df["Type"], df['Machine failure'])

awaryjnosc = pd.crosstab(df["Type"], df["Machine failure"], normalize ="index") * 100

display(crosstab)
display(awaryjnosc.round(2))

Typ maszyny a przyczyna awarii - zestawienie

In [ ]:
kolumny_awarii = ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

df.groupby('Type')[kolumny_awarii].sum()


Analiza rekordów z więcej niż jedną przyczyną awarii

In [ ]:
print(f"Ilość rekordów: {df[df[['TWF', 'HDF', 'PWF', 'OSF', 'RNF']].sum(axis=1) >= 2].shape[0]}")
display(df[df[["TWF", "HDF", "PWF", "OSF", "RNF"]].sum(axis=1) >= 2])
display(df[df[["TWF", "HDF", "PWF", "OSF", "RNF"]].sum(axis=1) >= 3])


Macierz korelacji parametrów procesowych

In [ ]:
corr_matrix = df[['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'Power [W]','Temperature difference']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap = 'coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Macierz korelacji parametrów procesowych")
plt.show()

Wykres punktowy zależności między Rotational speed [rpm] i Torque [Nm] a Machine failure

In [ ]:
sns.set_theme(style="ticks")

g = sns.FacetGrid(df, col="Type", hue = "Machine failure", palette={0: "lightgray", 1: "red"}, height=5, aspect = 1)

g.map(sns.scatterplot, "Rotational speed [rpm]", "Torque [Nm]", alpha=0.7)

g.add_legend(title="Awaria (1 = Tak)")
g.set_titles("Model maszyny: {col_name}")
plt.subplots_adjust(top=0.8)
g.fig.suptitle('Kiedy dochodzi do awarii (czerwone punkty) w zależności od typu maszyny', fontsize=16)

plt.show()

#### Analiza TWF

Analiza zależności między Tool wear [min], a TWF

In [ ]:
plot_box_hist(df, 'TWF', 'Tool wear [min]')

In [ ]:
print(f"Dolna granica przedziału: {df[df['TWF']==1]['Tool wear [min]'].agg('min')}")
print(f"Górna granica przedziału: {df[df['TWF']==1]['Tool wear [min]'].agg('max')}")

Analiza zależności między Rotational speed [rpm] a TWF

In [ ]:
plot_box_hist(df, 'TWF', 'Rotational speed [rpm]')

Analiza zależności między Torque [Nm], a TWF

In [ ]:
plot_box_hist(df, 'TWF', 'Torque [Nm]')

In [ ]:
sns.pairplot(df[['Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'TWF']], hue='TWF')

#### Analiza HDF

Analiza zależności między różnicą temperatur a HDF

In [ ]:
plot_box_hist(df, 'HDF', 'Temperature difference')

Analiza przedziału

In [ ]:
print(f"Dolna granica przedziału: {df[df['HDF'] == 1]['Temperature difference'].agg('min').round(2)}")
print(f"Górna granica przedziału: {df[df['HDF'] == 1]['Temperature difference'].agg('max').round(2)}")

Analiza zależności między Rotational speed [rpm] a HDF

In [ ]:
plot_box_hist(df, 'HDF', 'Rotational speed [rpm]')

Analiza przedziału

In [ ]:
print(f"Dolna granica przedziału: {df[df['HDF'] == 1]['Rotational speed [rpm]'].agg('min').round(2)}")
print(f"Górna granica przedziału: {df[df['HDF'] == 1]['Rotational speed [rpm]'].agg('max').round(2)}")

Analiza zależności między Power a PWF

#### Analiza PWF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.violinplot(x="PWF", y="Power [W]", data=df, palette="Set3", ax=axes[0], hue ="PWF", legend=False)
axes[0].set_title("Wykres wiolinowy PWF vs Power") 

# 3. Drugi wykres wrzucamy na axes[1] (prawa strona)
sns.histplot(data=df, x="Power [W]", hue="PWF", kde=True, ax=axes[1])
axes[1].set_title("Histogram mocy") 

plt.tight_layout()
plt.show()

Sprawdzenie punktu podziału

In [ ]:
from sklearn.cluster import KMeans

# Przygotowujemy dane (K-Means wymaga formatu DataFrame/2D, stąd podwójny nawias)
X_pwf = df[df['PWF'] == 1][['Power [W]']]

# Tworzymy model K-Means dla 2 grup
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
X_pwf['Szczyt'] = kmeans.fit_predict(X_pwf)

# Grupujemy po wyznaczonych szczytach i wyciągamy min oraz max
przedzialy_pwf = X_pwf.groupby('Szczyt')['Power [W]'].agg(['min', 'max']).sort_values(by='min')

# Zmieniamy nazwy indeksów na bardziej czytelne
przedzialy_pwf.index = ['Przedział1', 'Przedział2']
display(przedzialy_pwf.round(2))

#### Analiza OSF

Wyznaczenie wartości progowej

In [ ]:
kryterium_osf = df['Tool wear [min]'] * df['Torque [Nm]']
wartosc_progowa = kryterium_osf[df['OSF'] == 1].min()
print(f"Wyznaczona wartość progowa to około: {wartosc_progowa}")

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df,
    x='Tool wear [min]',
    y='Torque [Nm]',
    hue='OSF',  
    size='Power [W]',
    sizes=(20, 300),
    alpha=0.7,
    palette={0: '#a1c9f4', 1: '#ffb3bd'}
)

plt.title('Analiza awarii OSF w przestrzeni wielowymiarowej', fontsize=14, pad=15)
plt.xlabel('Tool wear [min]', fontsize=12)
plt.ylabel('Torque [Nm]', fontsize=12)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)

x_line = np.linspace(150, 250, 100)
y_line = wartosc_progowa / x_line

plt.plot(x_line, y_line, color='red', linestyle='--', linewidth=2, label='Teoretyczna granica OSF')
plt.tight_layout()
plt.show()